In [4]:
from functools import lru_cache
import os
import MeCab
from langchain_community.document_loaders import TextLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

def load_documents(docs_path="docs"):
    print(f"Loading documents from {docs_path}...")

    if not os.path.exists(docs_path):
        raise FileNotFoundError(f"The directory {docs_path} does not exist.")

    # Load all .txt files from the docs directory
    loader = DirectoryLoader(
        path=docs_path,
        glob="*.txt",
        loader_cls=TextLoader,
        loader_kwargs={"encoding": "utf-8"}
    )
    documents = loader.load()
    if len(documents) == 0:
        raise FileNotFoundError(f"No .txt files found in {docs_path}. Please add your company documents.")
    # Add the media_type and clean the source metadata
    for doc in documents:
        path = doc.metadata["source"]

        # Extract folder name (e.g., 'anime') and filename
        # This splits the path into pieces
        parts = path.split(os.sep)

        # 'parts[-2]' is the folder name, 'parts[-1]' is the filename
        doc.metadata["media_type"] = parts[-2]
        doc.metadata["source"] = parts[-1]
    return documents

# Initialize the Tagger
# unidic-lite is automatically detected by mecab-python3
tagger = MeCab.Tagger()

def mecab_len(text):
    """Counts the number of tokens in the text using MeCab."""
    node = tagger.parseToNode(text)
    count = 0
    while node:
        if node.surface:
            count += 1
        node = node.next
    return count

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,           # Limit chunks to 100 MeCab tokens
    chunk_overlap=150,         # 20 token overlap
    length_function=mecab_len, # Use our custom MeCab counter
    separators=["\n\n", "\n", "。", "、", " ", ""] # Japanese-friendly separators
)

def split_documents(documents):
    chunks = text_splitter.split_documents(documents)
    return chunks

def deduplicate_by_filename(docs):
    unique_docs = {}

    for doc in docs:
        filename = os.path.basename(doc.metadata["source"])
        current_type = doc.metadata.get("media_type", "unknown")

        if filename not in unique_docs:
            # First time seeing this file: store the doc
            unique_docs[filename] = doc
            # Convert the single string into a set of types
            unique_docs[filename].metadata["media_type"] = {current_type}
        else:
            # Already exists: add the new media_type to the set
            unique_docs[filename].metadata["media_type"].add(current_type)

    # Convert sets back to comma-separated strings for Pinecone compatibility
    final_docs = list(unique_docs.values())
    for doc in final_docs:
        # Join types: e.g., {"anime", "games"} -> "anime, games"
        doc.metadata["media_type"] = ", ".join(sorted(doc.metadata["media_type"]))

    print(f"Deduplication complete. Remaining unique documents: {len(final_docs)}")
    return final_docs




## read and dedup

In [5]:
folders = ('anime', 'manga', 'game', 'light_novel')
# folders = ('anime','manga', 'light_novel')
all_docs = []
for folder in folders:
    all_docs.extend(load_documents(f"/wiki_data/{folder}"))
print("before dedup", len(all_docs))
all_docs = deduplicate_by_filename(all_docs)
print("after dedup", len(all_docs))


Loading documents from /wiki_data/anime...
Loading documents from /wiki_data/manga...
Loading documents from /wiki_data/game...
Loading documents from /wiki_data/light_novel...
before dedup 37311
Deduplication complete. Remaining unique documents: 33758
after dedup 33758


In [6]:
for doc in all_docs:
    if ',' in doc.metadata['media_type']:
        print(doc.metadata['media_type'], doc.metadata['source'])
        break

anime, manga 07-GHOST.txt


In [7]:
split_docs = split_documents(all_docs)

In [8]:
split_docs[0]

Document(metadata={'source': '001_7親指トム.txt', 'media_type': 'anime'}, page_content='『001/7おや指トム』（ゼロゼロななぶんのいちおやゆびトム）（英語表記：TOM of T.H.U.M.B.）は、アメリカのビデオクラフト社と日本の東映動画による日米合作のテレビアニメである。全24話。\nタイトル表記について、『親指トム』や『親ゆびトム』と表記している文献やレコードが多いが、『おや指トム』が正しい。\n日本では、アニメ『キングコング』とのセットで放送。NET（現・テレビ朝日）系列局で毎週水曜 19:30 - 20:00 （日本標準時）に放送されていた。番組自体は全26回で、1967年4月5日から同年10月4日まで放送されていたが、ラスト2回を『キングコング』の放送に使うため、本作は同年9月20日放送分をもって終了した。\n本作の主人公は、名探偵のヒーローという設定である。対抗する悪の組織として、MAD（またまた悪事同盟）という組織が登場する。\n\nストーリー\n主人公のトムとその助手のジャックが、ちびっ子光線を浴びて小人化してしまった。小人化したことにより、普通の人では解決し得ない難事件を解決していく。\n\n声の出演（日本語吹き替え版）\nトム（主人公） - 近石真介\nジャック（トムの助手） - 八代駿\nチーフ（トムとジャックの上司） - 熊倉一雄\nジェフ・ブリッジス - 寺田誠\n千葉耕市\n槐柳二\n塚田正昭\n\n主題歌（日本語吹き替え版）\n「001/7おや指トム」\n作詞・作曲・編曲 - 小林亜星 / 歌 - フォー・シンガーズ / 発売元 - テイチク（現・テイチクエンタテインメント）\nこの曲が流れるパートの映像は、冒頭のチアガールが持つタイトル部と最後のタイトル表示以外は英語版でも同じである。\n「キンダーレコード」（日本グラモフォン）から発売されたカバー版では、ハニーナイツが歌唱を担当。フォー・シンガーズ版よりもテンポが遅く、後年のCDには収録されていない2・3番や間奏が存在する。\n\n各話リスト（日本語吹き替え版）\n参考：『東映動画アーカイブス にっぽんアニメの原点』ワールドフォトプレス、2010年、151 - 152頁。\n\n補足\n『キングコン

In [9]:
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key=os.environ.get("PINECONE_API_KEY"))

# Create the index if it doesn't exist
index_name = "japanese-wiki-index"

if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=1024, # Critical: 1024 for Voyage Multilingual 2
        metric="cosine", # Best for Voyage/OpenAI
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1" # Or your preferred region
        )
    )

In [19]:
import os
from dotenv import load_dotenv
from langchain_voyageai import VoyageAIEmbeddings
from langchain_pinecone import PineconeVectorStore
from tqdm import tqdm
from pinecone import Pinecone

load_dotenv()

# Initialize the direct Pinecone client
pc = Pinecone(api_key=os.environ.get("PINECONE_API_KEY"))

# 1. Initialize Voyage (The bilingual magic)
embeddings = VoyageAIEmbeddings(
    voyage_api_key=os.environ.get("VOYAGE_API_KEY"),
    model="voyage-4-lite"
)

# 2. Connect to Pinecone
vector_store = PineconeVectorStore(
    index_name="japanese-wiki-index",
    embedding=embeddings,
    pinecone_api_key=os.environ.get("PINECONE_API_KEY")
)

# 3. Upload with a simple loop for safety
# (split_docs is your list of 270k+ chunks)
batch_size = 100
for i in tqdm(range(0, len(split_docs), batch_size), desc="Uploading to Pinecone"):
    batch = split_docs[i : i + batch_size]
    vector_store.add_documents(batch)

    # Optional: Save progress to a file
    with open("progress.txt", "w") as f:
        f.write(str(i + batch_size))

print("All 33,771 articles are now searchable!")

Uploading to Pinecone: 100%|██████████| 2290/2290 [1:43:18<00:00,  2.71s/it]  

All 33,771 articles are now searchable!


In [18]:
#index = pc.Index(index_name)
#index.delete(delete_all=True)

{}

In [21]:
# Simple Test Search
query = "Tell me about the protagonist of Frieren" # Ask in English!

# Embed the query (Notice input_type="query")
query_emb = vo.embed([query], model="voyage-4-lite", input_type="query").embeddings[0]

# Search Pinecone
results = index.query(vector=query_emb, top_k=3, include_metadata=True)

for res in results["matches"]:
    print(f"\nScore: {res['score']:.4f} | Source: {res['metadata']['source']}")
    print(f"Content: {res['metadata']['text'][:200]}...")

NameError: name 'voyageai' is not defined